# 10 — Prophet Optional Forecast

**Gold Nexus Alpha — professor-style forecasting pipeline**

Purpose of this notebook:

1. Load the long univariate gold dataset created by Notebook 03.
2. Convert the dataset to Prophet format using `ds` for date and `y` for gold price.
3. Plot and inspect the time series before fitting, matching the professor's Prophet reference flow.
4. Use the locked time-based train / validation / test windows.
5. Fit a small, professor-safe Prophet candidate set.
6. Select the best candidate by validation RMSE.
7. Refit the selected Prophet configuration on train + validation, then evaluate on the test period.
8. Export Prophet model metrics, forecast path, and page JSON artifacts.
9. Push artifacts to GitHub.

Professor reference style mirrored:

- `pip install prophet`
- `from prophet import Prophet`
- rename date column to `ds`
- rename target column to `y`
- fit model
- create future dataframe / prediction dataframe
- predict `yhat`, `yhat_lower`, `yhat_upper`
- plot actual vs forecast
- add vertical split lines


In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the same clean Colab → GitHub artifact workflow as prior notebooks.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset: remove old clone if present.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

# Clone URL. Use token for private/push access, but never print the token.
public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

# Git identity for Colab commits.
run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))

# Ensure push remote uses token when available.
if GITHUB_TOKEN:
    run_cmd(
        ["git", "remote", "set-url", "origin", auth_url],
        cwd=str(PROJECT_DIR),
        display_cmd=["git", "remote", "set-url", "origin", f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git"]
    )

print("✅ Repository ready:", PROJECT_DIR)


In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load Prophet and standard time-series evaluation tools.
# - Install Prophet if the Colab runtime does not already have it.
# ======================================================================================

import sys
import json
import math
import glob
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# Professor reference uses: pip install prophet
try:
    from prophet import Prophet
    print("✅ Prophet already installed.")
except Exception:
    print("📦 Prophet not found. Installing Prophet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "prophet"])
    from prophet import Prophet
    print("✅ Prophet installed and imported.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("✅ Dependencies loaded.")


In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Notebook 03 output: data/aligned/model_ready_univariate.csv
# - Fall back to weekday_clean_matrix.csv or an uploaded Gold_Matrix CSV only if needed.
# - Lock the professor-safe univariate dataset and time splits.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Locked project rules.
OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

LONG_UNIVARIATE_START = "1968-01-04"
LONG_UNIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "1968-01-04"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

def find_input_file():
    """Find the most appropriate input file for Notebook 10."""
    candidates = [
        ALIGNED_DIR / "model_ready_univariate.csv",
        ALIGNED_DIR / "weekday_clean_matrix.csv",
    ]

    # Also allow Colab-uploaded files as fallback.
    upload_patterns = [
        "/content/model_ready_univariate*.csv",
        "/content/weekday_clean_matrix*.csv",
        "/content/Gold_Matrix*.csv",
    ]
    for pattern in upload_patterns:
        candidates.extend(Path(p) for p in glob.glob(pattern))

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find model_ready_univariate.csv, weekday_clean_matrix.csv, or uploaded Gold_Matrix CSV. "
        "Run Notebooks 01–03 first or upload the matrix CSV."
    )

INPUT_PATH = find_input_file()
print("✅ Input file detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
print("Raw input shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")
if "gold_price" not in raw_df.columns:
    raise ValueError("Input file must contain a 'gold_price' column.")

# Standardize date and sort.
df = raw_df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df = df.sort_values("date").drop_duplicates(subset=["date"], keep="last")

# If fallback is raw matrix, enforce weekday cleaning here.
df["weekday"] = df["date"].dt.dayofweek
if df["weekday"].isin([5, 6]).any():
    before = len(df)
    df = df[~df["weekday"].isin([5, 6])].copy()
    after = len(df)
    print(f"⚠️ Fallback cleaning applied: removed {before - after:,} Saturday/Sunday rows.")

# Lock official univariate window and target.
df = df[(df["date"] >= pd.Timestamp(LONG_UNIVARIATE_START)) & (df["date"] <= pd.Timestamp(LONG_UNIVARIATE_END))].copy()
df["gold_price"] = pd.to_numeric(df["gold_price"], errors="coerce")
df = df.dropna(subset=["gold_price"]).copy()

# Assign locked split labels if not already present.
def assign_split(d):
    d = pd.Timestamp(d)
    if pd.Timestamp(TRAIN_START) <= d <= pd.Timestamp(TRAIN_END):
        return "train"
    if pd.Timestamp(VALIDATION_START) <= d <= pd.Timestamp(VALIDATION_END):
        return "validation"
    if pd.Timestamp(TEST_START) <= d <= pd.Timestamp(TEST_END):
        return "test"
    return "outside_model_window"

df["split"] = df["date"].apply(assign_split)

model_df = df[df["split"].isin(["train", "validation", "test"])][["date", "gold_price", "split"]].copy()
model_df = model_df.sort_values("date").reset_index(drop=True)

# Prophet requires ds and y column names.
prophet_df = model_df.rename(columns={"date": "ds", "gold_price": "y"}).copy()

train_df = prophet_df[prophet_df["split"] == "train"][["ds", "y"]].copy()
valid_df = prophet_df[prophet_df["split"] == "validation"][["ds", "y"]].copy()
test_df  = prophet_df[prophet_df["split"] == "test"][["ds", "y"]].copy()

train_valid_df = pd.concat([train_df, valid_df], ignore_index=True).sort_values("ds")

print("✅ Model dataframe shape:", model_df.shape)
print(model_df["split"].value_counts())
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())

display(model_df.head())
display(model_df.tail())

# Professor-style plot before modeling.
plt.figure(figsize=(14, 6))
plt.plot(model_df["date"], model_df["gold_price"], linewidth=1)
plt.axvline(pd.Timestamp(TRAIN_END), linestyle="--", label="Train End")
plt.axvline(pd.Timestamp(VALIDATION_END), linestyle="--", label="Validation End")
plt.title("Gold Price Time Series — Long Univariate Dataset")
plt.xlabel("Date")
plt.ylabel("Gold Price")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
# ======================================================================================
# CELL 4 — PROPHET MODELING LOGIC
# ======================================================================================
# Purpose:
# - Fit a small, professor-safe Prophet candidate set.
# - Validate on 2019–2022.
# - Select best candidate by validation RMSE.
# - Refit selected configuration on train + validation.
# - Evaluate on 2023–2026 test period.
# ======================================================================================

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

def make_json_safe(obj):
    """Convert numpy/pandas objects into JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, (pd.Timestamp,)):
        return obj.strftime("%Y-%m-%d")
    if isinstance(obj, (datetime,)):
        return obj.isoformat()
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple)) else False:
        return None
    return obj

def metrics_dict(actual, predicted):
    """Professor-style accuracy metrics: ME, MAE, MSE, RMSE, MAPE, bias."""
    a = pd.Series(actual).astype(float).reset_index(drop=True)
    p = pd.Series(predicted).astype(float).reset_index(drop=True)
    mask = a.notna() & p.notna() & np.isfinite(a) & np.isfinite(p)
    a = a[mask]
    p = p[mask]
    if len(a) == 0:
        return {
            "n": 0,
            "mean_error": None,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "bias_direction": "not_available",
        }

    errors = a - p
    me = float(errors.mean())
    mae = float(mean_absolute_error(a, p))
    mse = float(mean_squared_error(a, p))
    rmse = float(math.sqrt(mse))

    nonzero = a != 0
    mape = float((np.abs((a[nonzero] - p[nonzero]) / a[nonzero])).mean() * 100) if nonzero.any() else None

    if me > 0:
        bias_direction = "under_forecast_on_average"
    elif me < 0:
        bias_direction = "over_forecast_on_average"
    else:
        bias_direction = "approximately_unbiased"

    return {
        "n": int(len(a)),
        "mean_error": me,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape,
        "bias_direction": bias_direction,
    }

def create_prophet_model(config):
    """Create Prophet model from a compact candidate configuration."""
    model = Prophet(
        growth=config.get("growth", "linear"),
        yearly_seasonality=config.get("yearly_seasonality", True),
        weekly_seasonality=config.get("weekly_seasonality", True),
        daily_seasonality=config.get("daily_seasonality", False),
        seasonality_mode=config.get("seasonality_mode", "additive"),
        changepoint_prior_scale=config.get("changepoint_prior_scale", 0.05),
        seasonality_prior_scale=config.get("seasonality_prior_scale", 10.0),
        interval_width=config.get("interval_width", 0.80),
    )
    return model

def fit_predict_prophet(train_data, predict_dates, config):
    """
    Fit Prophet on train_data and forecast exact dates in predict_dates.

    For log candidates:
    - fit Prophet to log(y)
    - exponentiate yhat, yhat_lower, yhat_upper back to price scale
    """
    train_data = train_data[["ds", "y"]].copy()
    predict_dates = pd.DataFrame({"ds": pd.to_datetime(pd.Series(predict_dates))}).drop_duplicates().sort_values("ds")

    if config.get("target_transform") == "log":
        fit_data = train_data.copy()
        fit_data["y"] = np.log(fit_data["y"])
        model = create_prophet_model(config)
        model.fit(fit_data)
        forecast = model.predict(predict_dates)
        for col in ["yhat", "yhat_lower", "yhat_upper"]:
            forecast[col] = np.exp(forecast[col])
    else:
        model = create_prophet_model(config)
        model.fit(train_data)
        forecast = model.predict(predict_dates)

    return model, forecast

# Small candidate set keeps the notebook practical and professor-safe.
PROPHET_CANDIDATES = [
    {
        "candidate_name": "Prophet_Default_Additive",
        "target_transform": "none",
        "growth": "linear",
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": False,
        "seasonality_mode": "additive",
        "changepoint_prior_scale": 0.05,
        "seasonality_prior_scale": 10.0,
        "interval_width": 0.80,
    },
    {
        "candidate_name": "Prophet_Flexible_Trend_Additive",
        "target_transform": "none",
        "growth": "linear",
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": False,
        "seasonality_mode": "additive",
        "changepoint_prior_scale": 0.20,
        "seasonality_prior_scale": 10.0,
        "interval_width": 0.80,
    },
    {
        "candidate_name": "Prophet_Multiplicative_Seasonality",
        "target_transform": "none",
        "growth": "linear",
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": False,
        "seasonality_mode": "multiplicative",
        "changepoint_prior_scale": 0.10,
        "seasonality_prior_scale": 10.0,
        "interval_width": 0.80,
    },
    {
        "candidate_name": "Prophet_Log_Target_Flexible",
        "target_transform": "log",
        "growth": "linear",
        "yearly_seasonality": True,
        "weekly_seasonality": True,
        "daily_seasonality": False,
        "seasonality_mode": "additive",
        "changepoint_prior_scale": 0.20,
        "seasonality_prior_scale": 10.0,
        "interval_width": 0.80,
    },
]

print("STEP 4 — Prophet candidate comparison")
print(f"Training rows: {len(train_df):,}")
print(f"Validation rows: {len(valid_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Candidates: {len(PROPHET_CANDIDATES)}")

candidate_results = []
validation_forecasts = {}

for idx, config in enumerate(PROPHET_CANDIDATES, start=1):
    candidate_name = config["candidate_name"]
    print(f"\n[{idx}/{len(PROPHET_CANDIDATES)}] Fitting {candidate_name} on train and forecasting validation...")

    model, forecast_valid = fit_predict_prophet(train_df, valid_df["ds"], config)

    merged_valid = valid_df.merge(
        forecast_valid[["ds", "yhat", "yhat_lower", "yhat_upper"]],
        on="ds",
        how="left"
    )

    valid_metrics = metrics_dict(merged_valid["y"], merged_valid["yhat"])
    print(f"Validation RMSE: {valid_metrics['rmse']:.4f} | MAE: {valid_metrics['mae']:.4f} | MAPE: {valid_metrics['mape']:.4f}%")

    candidate_record = {
        "candidate_name": candidate_name,
        "config": config,
        "validation_metrics": valid_metrics,
    }
    candidate_results.append(candidate_record)
    validation_forecasts[candidate_name] = merged_valid.copy()

candidate_results_sorted = sorted(
    candidate_results,
    key=lambda x: float("inf") if x["validation_metrics"]["rmse"] is None else x["validation_metrics"]["rmse"]
)

selected_candidate = candidate_results_sorted[0]
selected_config = selected_candidate["config"]
selected_name = selected_candidate["candidate_name"]

print("\n✅ Selected Prophet candidate by validation RMSE:")
print(selected_name)
print(selected_candidate["validation_metrics"])

# Refit selected configuration on train + validation and evaluate test period.
print(f"\nRefitting selected model on train + validation for final test evaluation: {selected_name}")
selected_model_train_valid, forecast_test = fit_predict_prophet(train_valid_df, test_df["ds"], selected_config)

merged_test = test_df.merge(
    forecast_test[["ds", "yhat", "yhat_lower", "yhat_upper"]],
    on="ds",
    how="left"
)
test_metrics = metrics_dict(merged_test["y"], merged_test["yhat"])
print("✅ Test metrics:")
print(test_metrics)

# Also keep selected validation forecast from train-only model for validation chart.
selected_validation = validation_forecasts[selected_name].copy()
validation_metrics = selected_candidate["validation_metrics"]

# Build combined forecast path.
path_train = train_df.copy()
path_train["split"] = "train"
path_train["actual"] = path_train["y"]
path_train["yhat"] = np.nan
path_train["yhat_lower"] = np.nan
path_train["yhat_upper"] = np.nan
path_train["forecast_stage"] = "actual_training_history"

path_valid = selected_validation.copy()
path_valid["split"] = "validation"
path_valid["actual"] = path_valid["y"]
path_valid["forecast_stage"] = "validation_forecast_fit_on_train"

path_test = merged_test.copy()
path_test["split"] = "test"
path_test["actual"] = path_test["y"]
path_test["forecast_stage"] = "test_forecast_fit_on_train_plus_validation"

forecast_path_df = pd.concat([
    path_train[["ds", "split", "actual", "yhat", "yhat_lower", "yhat_upper", "forecast_stage"]],
    path_valid[["ds", "split", "actual", "yhat", "yhat_lower", "yhat_upper", "forecast_stage"]],
    path_test[["ds", "split", "actual", "yhat", "yhat_lower", "yhat_upper", "forecast_stage"]],
], ignore_index=True).sort_values("ds")

forecast_path_df = forecast_path_df.rename(columns={"ds": "date"})

# Residuals for validation and test.
selected_validation["residual"] = selected_validation["y"] - selected_validation["yhat"]
merged_test["residual"] = merged_test["y"] - merged_test["yhat"]

# Professor-style plots.
plt.figure(figsize=(14, 6))
plt.plot(model_df["date"], model_df["gold_price"], label="Actual", linewidth=1)
plt.plot(selected_validation["ds"], selected_validation["yhat"], label="Prophet Validation Forecast", linewidth=1)
plt.plot(merged_test["ds"], merged_test["yhat"], label="Prophet Test Forecast", linewidth=1)
plt.fill_between(
    merged_test["ds"],
    merged_test["yhat_lower"].astype(float),
    merged_test["yhat_upper"].astype(float),
    alpha=0.15,
    label="Test Forecast Interval"
)
plt.axvline(pd.Timestamp(TRAIN_END), linestyle="--", label="Train End")
plt.axvline(pd.Timestamp(VALIDATION_END), linestyle="--", label="Validation End")
plt.title(f"Prophet Forecast — Selected Candidate: {selected_name}")
plt.xlabel("Date")
plt.ylabel("Gold Price")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(selected_validation["ds"], selected_validation["residual"], label="Validation Residuals", linewidth=1)
plt.plot(merged_test["ds"], merged_test["residual"], label="Test Residuals", linewidth=1)
plt.axhline(0, linestyle="--")
plt.title("Prophet Residuals")
plt.xlabel("Date")
plt.ylabel("Actual - Forecast")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Components plot for selected train+validation model, using its test forecast object.
try:
    fig_components = selected_model_train_valid.plot_components(forecast_test)
    plt.show()
except Exception as e:
    print("⚠️ Prophet components plot skipped:", e)

# Prepare small tables for display.
candidate_table = pd.DataFrame([
    {
        "candidate": r["candidate_name"],
        "validation_rmse": r["validation_metrics"]["rmse"],
        "validation_mae": r["validation_metrics"]["mae"],
        "validation_mape": r["validation_metrics"]["mape"],
        "changepoint_prior_scale": r["config"]["changepoint_prior_scale"],
        "seasonality_mode": r["config"]["seasonality_mode"],
        "target_transform": r["config"]["target_transform"],
    }
    for r in candidate_results_sorted
])
display(candidate_table)

display(forecast_path_df.tail())


In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Prophet results, forecast path, and frontend page JSON.
# ======================================================================================

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(payload), f, indent=2)
    print("✅ Wrote", path)

def write_csv(path, dataframe):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(path, index=False)
    print("✅ Wrote", path)

prophet_results = {
    "artifact_name": "prophet_results",
    "notebook": "10_prophet_optional_forecast.ipynb",
    "model_family": "Prophet",
    "model_status": "optional_benchmark",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "project_rules": {
        "dataset": "Dataset A — Long Univariate",
        "target": "gold_price",
        "official_forecast_cutoff_date": OFFICIAL_FORECAST_CUTOFF_DATE,
        "weekend_handling": "weekday_clean_from_prior_notebooks",
        "high_yield_used": False,
        "exogenous_variables_used": False,
        "selection_rule": "best_validation_rmse_among_small_professor_safe_candidate_set"
    },
    "professor_reference_style": {
        "uses_prophet_column_names": True,
        "date_column": "ds",
        "target_column": "y",
        "fit_predict_flow": [
            "rename date to ds",
            "rename gold_price to y",
            "fit Prophet model",
            "predict future/holdout dates",
            "plot forecast",
            "evaluate errors"
        ]
    },
    "time_windows": {
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": int(len(train_df))},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": int(len(valid_df))},
        "test": {"start": TEST_START, "end": TEST_END, "rows": int(len(test_df))}
    },
    "candidate_results": candidate_results_sorted,
    "selected_candidate": {
        "candidate_name": selected_name,
        "config": selected_config,
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics
    },
    "interpretation": [
        "Prophet is included as an optional recognizable forecasting benchmark, not as the assumed winner.",
        "Candidate selection is based on validation RMSE before test evaluation.",
        "The selected Prophet configuration is refit on train plus validation data before test-period evaluation.",
        "Prophet can capture trend and recurring seasonality, but it may still lag sharp gold regime shifts because it is a univariate model and does not use macro drivers."
    ],
    "limitations": [
        "Prophet uses gold_price only in this notebook.",
        "It does not use real yields, USD index, VIX, geopolitical risk, policy uncertainty, oil, or ETF holdings.",
        "The long test period contains a large post-2023 gold surge, so Prophet may under-forecast if the surge is not represented in the fitted historical trend.",
        "Prophet is optional and should be judged only inside Notebook 11 model comparison."
    ]
}

# CSV path is not required in the original plan, but useful for debugging and frontend fallback.
prophet_forecast_path_csv = MODELS_ARTIFACTS_DIR / "prophet_forecast_path.csv"
write_csv(prophet_forecast_path_csv, forecast_path_df)

prophet_forecast_path_json = {
    "artifact_name": "prophet_forecast_path",
    "notebook": "10_prophet_optional_forecast.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "selected_candidate": selected_name,
    "records": forecast_path_df.to_dict(orient="records"),
    "columns": list(forecast_path_df.columns),
    "notes": [
        "Training rows contain actual history only.",
        "Validation forecast is produced from the model fit on training data.",
        "Test forecast is produced from the selected model refit on train plus validation data."
    ]
}

page_prophet = {
    "page_title": "Prophet Optional Forecast",
    "page_subtitle": "Univariate benchmark using Prophet on the long gold price history.",
    "artifact_sources": [
        "artifacts/models/prophet_results.json",
        "artifacts/models/prophet_forecast_path.json"
    ],
    "model_category": "optional_benchmark",
    "dataset_window": {
        "name": "Dataset A — Long Univariate",
        "start": LONG_UNIVARIATE_START,
        "end": LONG_UNIVARIATE_END,
        "target": "gold_price"
    },
    "training_period": {"start": TRAIN_START, "end": TRAIN_END},
    "validation_period": {"start": VALIDATION_START, "end": VALIDATION_END},
    "test_period": {"start": TEST_START, "end": TEST_END},
    "selected_model": {
        "name": selected_name,
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
        "config": selected_config
    },
    "kpi_cards": [
        {"label": "Validation RMSE", "value": validation_metrics.get("rmse"), "format": "number"},
        {"label": "Validation MAE", "value": validation_metrics.get("mae"), "format": "number"},
        {"label": "Validation MAPE", "value": validation_metrics.get("mape"), "format": "percent"},
        {"label": "Test RMSE", "value": test_metrics.get("rmse"), "format": "number"},
        {"label": "Test MAE", "value": test_metrics.get("mae"), "format": "number"},
        {"label": "Test MAPE", "value": test_metrics.get("mape"), "format": "percent"}
    ],
    "charts": [
        {
            "chart_id": "prophet_actual_vs_forecast",
            "title": "Actual vs Prophet Forecast",
            "data_artifact": "artifacts/models/prophet_forecast_path.json",
            "x": "date",
            "series": ["actual", "yhat", "yhat_lower", "yhat_upper"]
        }
    ],
    "model_explanation": [
        "Prophet decomposes a time series into trend, seasonality, and changepoint components.",
        "This notebook uses Prophet as an optional univariate benchmark on gold_price only.",
        "The best Prophet candidate is selected by validation RMSE, then evaluated on the test period."
    ],
    "limitations": prophet_results["limitations"],
    "professor_safe_note": "Prophet is not treated as the final model until Notebook 11 compares it against all other completed methods."
}

write_json(MODELS_ARTIFACTS_DIR / "prophet_results.json", prophet_results)
write_json(MODELS_ARTIFACTS_DIR / "prophet_forecast_path.json", prophet_forecast_path_json)
write_json(PAGES_ARTIFACTS_DIR / "page_prophet.json", page_prophet)

print("\n✅ Notebook 10 artifact export complete.")


In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 10 Prophet outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/prophet_results.json",
    "artifacts/models/prophet_forecast_path.json",
    "artifacts/models/prophet_forecast_path.csv",
    "artifacts/pages/page_prophet.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Add Notebook 10 Prophet optional forecast artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ Notebook 10 Prophet artifacts pushed to GitHub.")
else:
    print("ℹ️ No changes detected. Nothing to commit.")

print("✅ Notebook 10 complete.")
